In [15]:
# ============================================================
# Full Inference Script for 16 trained checkpoints
# Models : BERT, ALBERT, TinyBERT, MobileBERT
# Train  : Kia, Tesla, Genesis, Silverado
# Eval   : Kia, Tesla, Genesis, Silverado
# ============================================================

# =========================
# Imports
# =========================
import os
import gc
import glob
import random
import warnings
import numpy as np
import pandas as pd
import torch

from dotenv import load_dotenv
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)

from utils import SegmentFromFile

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

# =========================
# Environment + Seed
# =========================
load_dotenv()
hf_token = os.getenv("HF_TOKEN", None)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================
# Configuration
# =========================

CHECKPOINTS = {
    "BERT": {
        "Kia Soul": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/Bert/TrainKia",
        "Tesla Model 3": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/Bert/TrainTesla",
        "Genesis G80": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/Bert/TrainGen",
        "Chevrolet Silverado": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/Bert/TrainSil",
    },
    "ALBERT": {
        "Kia Soul": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/AlBert/TrainKia",
        "Tesla Model 3": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/AlBert/TrainTesla",
        "Genesis G80": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/AlBert/TrainGen",
        "Chevrolet Silverado": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/AlBert/TrainSil",
    },
    "TinyBERT": {
        "Kia Soul": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/TinyBert/TrainKia",
        "Tesla Model 3": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/TinyBert/TrainTesla",
        "Genesis G80": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/TinyBert/TrainGen",
        "Chevrolet Silverado": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/TinyBert/TrainSil",
    },
    "MobileBERT": {
        "Kia Soul": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/MobileBert/TrainKia/mobilebert-can-attack-classifier-experimental",
        "Tesla Model 3": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/MobileBert/TrainTesla/mobilebert-can-attack-classifier-experimental",
        "Genesis G80": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/MobileBert/TrainGen/mobilebert-can-attack-classifier-experimental",
        "Chevrolet Silverado": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/MobileBert/TrainSil/mobilebert-can-attack-classifier-experimental",
    },
}

BASE_MODELS = {
    "BERT": "google-bert/bert-base-uncased",
    "ALBERT": "albert/albert-base-v2",
    "TinyBERT": "nreimers/TinyBERT_L-4_H-312_v2",
    "MobileBERT": "google/mobilebert-uncased",
}

# Evaluation datasets
EVAL_DIRS = {
    "Test": "/home/lisa/Arupreza/UIDS/UIDS-II/Split_data/Val",
}

TIME_GAP_EVAL = 100.0
MAX_LENGTH = 512

OUTPUT_ROOT = "./inference_only_results_16"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# =========================
# Helper Functions
# =========================

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def format_chunk_to_string(chunk):
    tokens = []
    for pair in chunk:
        tokens.append(f"T{int(pair[0])}")
        tokens.append(f"G{int(pair[1])}")
    return " ".join(tokens)

def load_and_process_data(directory, time_gap):
    all_chunks, all_labels = [], []

    csv_files = sorted(glob.glob(os.path.join(directory, "*.csv")))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files in {directory}")

    for file_path in csv_files:
        filename = os.path.basename(file_path)
        chunks, labels = SegmentFromFile(directory, filename, time_gap=time_gap)

        all_chunks.extend(chunks)
        all_labels.extend(labels)

    texts = [format_chunk_to_string(chunk) for chunk in all_chunks]

    return pd.DataFrame({"text": texts, "label": all_labels})

def tokenize_dataset(df, tokenizer, max_length=512):

    ds = Dataset.from_pandas(df, preserve_index=False)

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding=False,
            max_length=max_length
        )

    ds = ds.map(tokenize_function, batched=True)

    cols_to_remove = [c for c in ["text", "__index_level_0__"] if c in ds.column_names]
    if cols_to_remove:
        ds = ds.remove_columns(cols_to_remove)

    ds = ds.rename_column("label", "labels")
    ds.set_format("torch")

    return ds

def compute_metrics_from_arrays(y_true, y_pred):

    avg = "binary" if len(set(y_true)) == 2 else "weighted"

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred,
        average=avg,
        zero_division=0
    )

    acc = accuracy_score(y_true, y_pred)

    return {
        "F1": float(f1),
        "Accuracy": float(acc),
        "Precision": float(precision),
        "Recall": float(recall),
    }

def load_tokenizer(checkpoint_path, base_model_name):

    if os.path.exists(os.path.join(checkpoint_path, "tokenizer_config.json")):
        return AutoTokenizer.from_pretrained(checkpoint_path, token=hf_token, use_fast=True)

    return AutoTokenizer.from_pretrained(base_model_name, token=hf_token, use_fast=True)

# =========================
# Load Evaluation Datasets
# =========================

eval_dataframes = {}

for vehicle, path in EVAL_DIRS.items():
    print("Loading eval set:", vehicle)
    eval_dataframes[vehicle] = load_and_process_data(path, TIME_GAP_EVAL)

# =========================
# Inference Loop
# =========================

all_results = []

for model_name, vehicle_ckpts in CHECKPOINTS.items():

    base_model = BASE_MODELS[model_name]

    for train_vehicle, checkpoint_path in vehicle_ckpts.items():

        print(f"\nModel: {model_name} | Train: {train_vehicle}")

        clear_memory()

        tokenizer = load_tokenizer(checkpoint_path, base_model)

        model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint_path,
            token=hf_token
        )

        args = TrainingArguments(
            output_dir=os.path.join(
                OUTPUT_ROOT,
                f"tmp_{model_name}_{train_vehicle.replace(' ','_')}"
            ),
            per_device_eval_batch_size=8,
            report_to="none",
            fp16=torch.cuda.is_available(),
        )

        trainer = Trainer(
            model=model,
            args=args,
            tokenizer=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        )

        for eval_vehicle, eval_df in eval_dataframes.items():

            print(f"   Evaluating on: {eval_vehicle}")

            eval_dataset = tokenize_dataset(eval_df, tokenizer, MAX_LENGTH)

            pred = trainer.predict(eval_dataset)

            y_true = pred.label_ids
            y_pred = pred.predictions.argmax(-1)

            metrics = compute_metrics_from_arrays(y_true, y_pred)

            all_results.append({
                "Model": model_name,
                "Train Vehicle": train_vehicle,
                "Eval Vehicle": eval_vehicle,
                "F1": metrics["F1"],
                "Accuracy": metrics["Accuracy"],
                "Precision": metrics["Precision"],
                "Recall": metrics["Recall"],
                "Samples": len(eval_df)
            })

        del trainer, model, tokenizer
        clear_memory()

# =========================
# Save Results
# =========================

results_df = pd.DataFrame(all_results)

csv_path = os.path.join(OUTPUT_ROOT, "full_results.csv")
results_df.to_csv(csv_path, index=False)

print("\nSaved results to:", csv_path)

# =========================
# Pivot table (paper format)
# =========================

pivot_f1 = results_df.pivot_table(
    index="Model",
    columns=["Train Vehicle", "Eval Vehicle"],
    values="F1"
)

print("\nF1 Score Table")
print(pivot_f1)

Loading eval set: Test

Model: BERT | Train: Kia Soul
   Evaluating on: Test


Map: 100%|██████████| 28998/28998 [00:29<00:00, 998.85 examples/s] 



Model: BERT | Train: Tesla Model 3
   Evaluating on: Test


Map: 100%|██████████| 28998/28998 [00:28<00:00, 1010.28 examples/s]



Model: BERT | Train: Genesis G80
   Evaluating on: Test


...


Model: MobileBERT | Train: Genesis G80
   Evaluating on: Test


Map: 100%|██████████| 28998/28998 [00:28<00:00, 1011.82 examples/s]



Model: MobileBERT | Train: Chevrolet Silverado
   Evaluating on: Test


Map: 100%|██████████| 28998/28998 [00:28<00:00, 1015.25 examples/s]



Saved results to: ./inference_only_results_16/full_results.csv

F1 Score Table
Train Vehicle Chevrolet Silverado Genesis G80  Kia Soul Tesla Model 3
Eval Vehicle                 Test        Test      Test          Test
Model                                                                
ALBERT                   0.981671    0.981671  0.981671      0.982799
BERT                     0.983941    0.983941  0.983938      0.983941
MobileBERT               0.962027    0.293403  0.984895      0.983990
TinyBERT                 0.915827    0.982442  0.983941      0.988183


In [1]:
import os
import gc
import glob
import time
import warnings
import numpy as np
import pandas as pd
import torch
import psutil
import pynvml

from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

from utils import SegmentFromFile

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

/home/lisa/Arupreza/UIDS-II/uids/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/lisa/Arupreza/UIDS-II/uids/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_cpu_usage():
    return psutil.cpu_percent(interval=None)

def get_cpu_memory():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024**2)

In [14]:
pynvml.nvmlInit()
gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)

def get_gpu_memory():
    mem = pynvml.nvmlDeviceGetMemoryInfo(gpu_handle)
    return mem.used / (1024**2)

def get_gpu_util():
    util = pynvml.nvmlDeviceGetUtilizationRates(gpu_handle)
    return util.gpu

In [15]:
TIME_GAP_EVAL = 100.0
MAX_LENGTH = 512

In [16]:
CHECKPOINTS = {
    "BERT": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/Bert/TrainKia",
    "ALBERT": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/AlBert/TrainKia",
    "TinyBERT": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/TinyBert/TrainKia",
    "MobileBERT": "/home/lisa/Arupreza/UIDS/UIDS-II/SFTSrc/MobileBert/TrainKia/mobilebert-can-attack-classifier-experimental"
}

In [17]:
BASE_MODELS = {
    "BERT": "google-bert/bert-base-uncased",
    "ALBERT": "albert/albert-base-v2",
    "TinyBERT": "nreimers/TinyBERT_L-4_H-312_v2",
    "MobileBERT": "google/mobilebert-uncased",
}

In [18]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [19]:
def format_chunk_to_string(chunk):

    tokens = []

    for pair in chunk:
        tokens.append(f"T{int(pair[0])}")
        tokens.append(f"G{int(pair[1])}")

    return " ".join(tokens)

In [20]:
def load_and_process_data(directory, time_gap):

    all_chunks = []
    all_labels = []

    csv_files = sorted(glob.glob(os.path.join(directory, "*.csv")))

    for file_path in csv_files:

        filename = os.path.basename(file_path)

        chunks, labels = SegmentFromFile(
            directory,
            filename,
            time_gap=time_gap
        )

        all_chunks.extend(chunks)
        all_labels.extend(labels)

    texts = [format_chunk_to_string(chunk) for chunk in all_chunks]

    return pd.DataFrame({
        "text": texts,
        "label": all_labels
    })

In [21]:
def tokenize_dataset(df, tokenizer):

    ds = Dataset.from_pandas(df, preserve_index=False)

    def tokenize(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=MAX_LENGTH
        )

    ds = ds.map(tokenize, batched=True)

    ds = ds.remove_columns(["text"])
    ds = ds.rename_column("label", "labels")
    ds.set_format("torch")

    return ds

In [22]:
def benchmark_model(model_name, checkpoint, base_model, eval_df):

    print(f"\nRunning Benchmark: {model_name}")

    clear_memory()

    cpu_before = get_cpu_usage()
    cpu_mem_before = get_cpu_memory()
    gpu_mem_before = get_gpu_memory()

    tokenizer = AutoTokenizer.from_pretrained(base_model)

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

    args = TrainingArguments(
        output_dir="./bench_tmp",
        per_device_eval_batch_size=8,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer)
    )

    start = time.time()

    dataset = tokenize_dataset(eval_df, tokenizer)

    trainer.predict(dataset)

    total_time = time.time() - start

    cpu_after = get_cpu_usage()
    cpu_mem_after = get_cpu_memory()
    gpu_mem_after = get_gpu_memory()
    gpu_util = get_gpu_util()

    samples = len(eval_df)

    latency = (total_time / samples) * 1000
    throughput = samples / total_time

    clear_memory()

    return {
        "Model": model_name,
        "Samples": samples,
        "Total Time (s)": total_time,
        "Latency (ms/sample)": latency,
        "Throughput (samples/sec)": throughput,
        "CPU Usage (%)": cpu_after,
        "CPU Memory (MB)": cpu_mem_after - cpu_mem_before,
        "GPU Memory (MB)": gpu_mem_after - gpu_mem_before,
        "GPU Util (%)": gpu_util
    }

In [23]:
data_path = "/home/lisa/Arupreza/UIDS/UIDS-II/Split_data/Val"

eval_df = load_and_process_data(data_path, TIME_GAP_EVAL)

print("Validation samples:", len(eval_df))

Validation samples: 28998


In [24]:
results = []

for model in CHECKPOINTS:

    res = benchmark_model(
        model,
        CHECKPOINTS[model],
        BASE_MODELS[model],
        eval_df
    )

    results.append(res)

benchmark_df = pd.DataFrame(results)

print("\nComputational Benchmark")
print(benchmark_df)


Running Benchmark: BERT


Map: 100%|██████████| 28998/28998 [00:28<00:00, 1027.32 examples/s]



Running Benchmark: ALBERT


Map: 100%|██████████| 28998/28998 [00:34<00:00, 840.98 examples/s]



Running Benchmark: TinyBERT


Map: 100%|██████████| 28998/28998 [00:28<00:00, 1025.06 examples/s]



Running Benchmark: MobileBERT


Map: 100%|██████████| 28998/28998 [00:28<00:00, 1018.31 examples/s]



Computational Benchmark
        Model  Samples  Total Time (s)  Latency (ms/sample)  \
0        BERT    28998      237.339998             8.184702   
1      ALBERT    28998      291.845165            10.064320   
2    TinyBERT    28998       74.707352             2.576293   
3  MobileBERT    28998      187.259097             6.457656   

   Throughput (samples/sec)  CPU Usage (%)  CPU Memory (MB)  GPU Memory (MB)  \
0                122.179153            5.3       840.710938         997.3125   
1                 99.360906            5.3       313.011719         396.0000   
2                388.154569            5.2       199.148438         348.0000   
3                154.854960            5.2        13.421875         286.0000   

   GPU Util (%)  
0            92  
1            97  
2            73  
3            92  
